# Training pipeline — Will a country's real military spending grow > 5% next year?

**Decision supported:** a defence contractor's international sales team decides which
countries to send a business-development team to for next year's procurement cycle.

**One row** = one country in year *t* (features known at the end of year *t*).
**Label** = 1 if real (inflation-adjusted, local-currency) military spending grows by
more than 5% from *t* to *t+1*.

Pipeline: load → build target → clean → engineer features → time split →
preprocess (fit on train only) → baseline + three logistic regressions → threshold by
business cost → save artifacts for `app.py`.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, log_loss, precision_score, recall_score,
                             roc_auc_score)
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = Path("data/raw.csv")
ART = Path("artifacts")
ART.mkdir(exist_ok=True)

## 1. Business parameters (decided in REPORT.md, they drive the threshold)

- **False positive** (team sent, no budget increase): cost of a wasted sales campaign.
- **False negative** (increase missed): lost expected margin on contracts a competitor wins.

A missed opportunity is assumed 2x as costly as a wasted campaign: missing the
signal doesn't lose the whole contract (you can still bid later), only part of the
margin. So the cost-optimal threshold should sit **below** 0.5 (theory: 1/3).

In [ ]:
GROWTH_THRESHOLD = 0.05      # label: real growth next year > 5%
COST_FP = 300_000            # EUR, wasted sales campaign
COST_FN = 600_000            # EUR, lost expected margin
TRAIN_YEARS = (1990, 2010)   # feature years
VAL_YEARS = (2011, 2016)
TEST_YEARS = (2017, 2023)
LIVE_YEAR = 2024             # no label yet -> "outlook for 2025" in the dashboard

## 2. Load data and build the target

In [ ]:
raw = pd.read_csv(DATA).sort_values(["iso3", "year"]).reset_index(drop=True)
print(raw.shape)

df = raw.copy()
g = df.groupby("iso3")

# Real growth = nominal local-currency growth deflated by the GDP deflator.
# Local currency avoids fake growth from exchange-rate swings; deflating avoids
# fake growth from inflation (e.g. Argentina, Turkey).
nominal_growth = g["milex_lcu"].pct_change(fill_method=None)
df["real_growth"] = (1 + nominal_growth) / (1 + df["gdp_deflator"] / 100) - 1
df["next_real_growth"] = df.groupby("iso3")["real_growth"].shift(-1)

### Data-quality rule
Growth values that are infinite (spending was 0) or extreme are almost always currency
redenominations or reporting breaks, not real changes. Rows whose **label** comes from
such a value are dropped, since the label would be wrong.

In [ ]:
PLAUSIBLE = (-0.6, 1.5)  # -60% .. +150% real growth in one year
lbl = df["next_real_growth"]
bad_label = ~np.isfinite(lbl) | (lbl < PLAUSIBLE[0]) | (lbl > PLAUSIBLE[1])
print("rows with implausible next-year growth:", int((bad_label & lbl.notna()).sum()))

# Same rule for the current-year growth used as a feature: set to NaN (imputed later)
cur = df["real_growth"]
df.loc[~np.isfinite(cur) | (cur < PLAUSIBLE[0]) | (cur > PLAUSIBLE[1]), "real_growth"] = np.nan

df["target"] = (df["next_real_growth"] > GROWTH_THRESHOLD).astype(float)
df.loc[lbl.isna() | bad_label, "target"] = np.nan

## 3. Feature engineering
Every feature uses only information available at the end of year *t*.
Four groups, each one a hypothesis about why a country raises its budget.

In [ ]:
g = df.groupby("iso3")

# --- Group A: military momentum (budgets are multi-year programmes) ---
df["real_growth_3y"] = g["real_growth"].transform(lambda s: s.rolling(3, min_periods=2).mean())
df["d_milex_gdp"] = g["milex_gdp"].diff()  # change in burden, percentage points

# --- Group B: economic capacity (you can only spend what you have) ---
df["log_gdp_pc"] = np.log(df["gdp_pc"])
df["log_population"] = np.log(df["population"])
df["inflation"] = df["gdp_deflator"].clip(-20, 100)  # hyperinflation tails capped
df["gdp_growth_3y"] = g["gdp_growth"].transform(lambda s: s.rolling(3, min_periods=2).mean())

# --- Group C: burden & room to grow (catch-up vs. saturation) ---
df["region_median_burden"] = df.groupby(["region", "year"])["milex_gdp"].transform("median")
df["burden_gap_region"] = df["milex_gdp"] - df["region_median_burden"]
df["below_2pct"] = (df["milex_gdp"] < 2).astype(float)
df["log_armed_forces_pc"] = np.log1p(df["armed_forces"] / df["population"] * 1000)

# --- Group D: neighbour / global threat (security dilemma) ---
# Leave-one-out mean so a country's own growth doesn't leak into its "neighbours".
grp = df.groupby(["region", "year"])["real_growth"]
s, n = grp.transform("sum"), grp.transform("count")
own = df["real_growth"].fillna(0)
own_n = df["real_growth"].notna().astype(int)
df["region_peer_growth"] = (s - own) / (n - own_n).replace(0, np.nan)
df["global_median_growth"] = df.groupby("year")["real_growth"].transform("median")

FEATURE_GROUPS = {
    "momentum": ["real_growth", "real_growth_3y", "d_milex_gdp"],
    "economy": ["gdp_growth", "gdp_growth_3y", "log_gdp_pc", "log_population",
                "inflation", "resource_rents", "trade_gdp"],
    "burden": ["milex_gdp", "milex_gov", "burden_gap_region", "below_2pct",
               "log_armed_forces_pc"],
    "neighbours": ["region_peer_growth", "global_median_growth"],
}
NUMERIC = [f for fs in FEATURE_GROUPS.values() for f in fs]

# Region as one-hot. NOTE: World Bank *income group* is deliberately NOT used — it is
# today's classification, so using it for 1995 would leak information from the future.
REGIONS = sorted(df["region"].unique())
for r in REGIONS:
    df[f"region_{r}"] = (df["region"] == r).astype(float)
REGION_COLS = [f"region_{r}" for r in REGIONS[1:]]  # drop first: avoid perfect collinearity

## 4. Time-based split
The sales team decides in year *t* using data up to *t*. A random split would let the
model learn from later years and inflate test scores, so we split by feature year.
Pre-1990 is excluded: Cold-War dynamics (and the bloc countries) are a different regime.

In [ ]:
def years(a, b):
    return df[(df["year"] >= a) & (df["year"] <= b) & df["target"].notna()
              & df["milex_gdp"].notna()].copy()

train, val, test = years(*TRAIN_YEARS), years(*VAL_YEARS), years(*TEST_YEARS)
live = df[(df["year"] == LIVE_YEAR) & df["milex_gdp"].notna()].copy()
for name, part in [("train", train), ("val", val), ("test", test), ("live", live)]:
    print(f"{name:5s} rows={len(part):5d}  positive rate={part['target'].mean():.3f}")

## 5. Preprocessing — fitted on TRAIN only
1. Winsorise each numeric feature at the train 1st/99th percentile (outliers would
   dominate a linear model's gradient).
2. Missing values → train median **plus** a `_missing` flag (missingness itself can be
   informative, e.g. countries that stop reporting).
3. Standardise (needed so one learning rate works for all features in PyTorch, and so
   the L2 penalty treats features equally).

In [ ]:
class Preprocessor:
    def fit(self, X: pd.DataFrame):
        self.low = X[NUMERIC].quantile(0.01)
        self.high = X[NUMERIC].quantile(0.99)
        self.median = X[NUMERIC].clip(self.low, self.high, axis=1).median()
        self.missing_cols = [c for c in NUMERIC if X[c].isna().any()]
        self.columns = NUMERIC + [f"{c}_missing" for c in self.missing_cols] + REGION_COLS
        self.scaler = StandardScaler().fit(self._raw(X))
        return self

    def _raw(self, X):
        out = X[NUMERIC].clip(self.low, self.high, axis=1)
        for c in self.missing_cols:
            out[f"{c}_missing"] = X[c].isna().astype(float)
        out = out.fillna(self.median)
        for c in REGION_COLS:
            out[c] = X[c]
        return out[self.columns].to_numpy(dtype=np.float64)

    def transform(self, X):
        return self.scaler.transform(self._raw(X))


prep = Preprocessor().fit(train)
X_tr, X_va, X_te, X_live = (prep.transform(p) for p in (train, val, test, live))
y_tr, y_va, y_te = (p["target"].to_numpy() for p in (train, val, test))
FEATURES = prep.columns
print("n features:", len(FEATURES))

## 6. Naive baselines
- **Majority class**: always predict "no increase" (probability = train positive rate).
- **Persistence**: predict "increase" if this year's real growth was > 5% (what a
  salesperson would guess from the news). A useful model must beat both.

In [ ]:
def evaluate(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": roc_auc_score(y, p) if len(np.unique(p)) > 1 else 0.5,
        "pr_auc": average_precision_score(y, p),
        "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)),
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "cost_eur": int(fp * COST_FP + fn * COST_FN),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }

p_majority = np.full(len(y_te), y_tr.mean())
p_persist = (test["real_growth"].fillna(0) > GROWTH_THRESHOLD).astype(float).to_numpy()

## 7. Model 1 — scikit-learn LogisticRegression
Only hyperparameter tuned: regularisation strength `C`, chosen on validation log-loss.
sklearn minimises  `C * sum(logloss) + 0.5 * ||w||²`  (bias not penalised).

In [ ]:
results_C = {}
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    m = LogisticRegression(C=C, max_iter=5000).fit(X_tr, y_tr)
    results_C[C] = log_loss(y_va, m.predict_proba(X_va)[:, 1])
    print(f"C={C:<6} val log-loss={results_C[C]:.4f}")
BEST_C = min(results_C, key=results_C.get)
sk_model = LogisticRegression(C=BEST_C, max_iter=5000).fit(X_tr, y_tr)
print("best C:", BEST_C)

# The same objective divided by n gives the mean-loss form used in PyTorch:
#   mean(logloss) + LAMBDA/2 * ||w||²   with  LAMBDA = 1 / (C * n)
LAMBDA = 1.0 / (BEST_C * len(y_tr))

## 8. Model 2 — PyTorch from scratch (raw tensors + autograd + manual update)
Same model (`sigmoid(Xw + b)`), same loss, same L2 penalty as sklearn. Full-batch
gradient descent; the learning rate is the only extra hyperparameter (chosen on val).

In [ ]:
Xt_tr = torch.tensor(X_tr, dtype=torch.float32)
yt_tr = torch.tensor(y_tr, dtype=torch.float32)
Xt_va = torch.tensor(X_va, dtype=torch.float32)
EPOCHS = 3000


def train_manual(lr, epochs=EPOCHS):
    torch.manual_seed(SEED)
    w = torch.zeros(Xt_tr.shape[1], requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    history = []
    for epoch in range(epochs):
        logits = Xt_tr @ w + b
        p = torch.sigmoid(logits)
        eps = 1e-7
        bce = -(yt_tr * torch.log(p + eps) + (1 - yt_tr) * torch.log(1 - p + eps)).mean()
        loss = bce + LAMBDA / 2 * (w ** 2).sum()
        loss.backward()                     # autograd fills w.grad, b.grad
        with torch.no_grad():               # manual gradient step
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        history.append(loss.item())
    return w.detach(), b.detach(), history


def manual_predict(w, b, X):
    with torch.no_grad():
        return torch.sigmoid(torch.tensor(X, dtype=torch.float32) @ w + b).numpy()


lr_scores = {}
for lr in [0.01, 0.1, 0.5]:
    w, b, _ = train_manual(lr)
    lr_scores[lr] = log_loss(y_va, manual_predict(w, b, X_va))
    print(f"lr={lr:<5} val log-loss={lr_scores[lr]:.4f}")
BEST_LR = min(lr_scores, key=lr_scores.get)
w_manual, b_manual, hist_manual = train_manual(BEST_LR)
print("best lr:", BEST_LR, " final train loss:", round(hist_manual[-1], 5))

## 9. Model 3 — standard PyTorch (`nn.Module` + `torch.optim`)
Same model, loss, penalty, optimiser type (plain SGD, full batch) and learning rate,
so any difference vs. Model 2 would point to a bug, not a modelling choice.
The L2 penalty is added explicitly on the weights only — `weight_decay` in the
optimiser would also shrink the bias, which sklearn does not do.

In [ ]:
class LogReg(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)  # logits; sigmoid lives in the loss


def train_standard(lr, epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = LogReg(Xt_tr.shape[1])
    torch.nn.init.zeros_(model.linear.weight)
    torch.nn.init.zeros_(model.linear.bias)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()  # numerically stable sigmoid + BCE
    history = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xt_tr), yt_tr) + LAMBDA / 2 * (model.linear.weight ** 2).sum()
        loss.backward()
        opt.step()
        history.append(loss.item())
    model.eval()
    return model, history


def standard_predict(model, X):
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X, dtype=torch.float32))).numpy()


std_model, hist_std = train_standard(BEST_LR)
print("final train loss:", round(hist_std[-1], 5))

## 10. Do the three implementations agree?
Same data, same objective → coefficients should match up to optimiser tolerance.

In [ ]:
coef = pd.DataFrame({
    "feature": FEATURES,
    "sklearn": sk_model.coef_[0],
    "torch_manual": w_manual.numpy(),
    "torch_standard": std_model.linear.weight.detach().numpy()[0],
})
coef.loc[len(coef)] = ["(intercept)", sk_model.intercept_[0], b_manual.item(),
                       std_model.linear.bias.item()]
coef["max_abs_diff"] = (coef[["sklearn", "torch_manual", "torch_standard"]]
                        .sub(coef["sklearn"], axis=0).abs().max(axis=1))
print(coef.sort_values("sklearn", key=abs, ascending=False).round(4).to_string(index=False))
print("\nlargest coefficient difference:", coef["max_abs_diff"].max().round(5))

# Why aren't they identical? sklearn solves the problem to convergence (L-BFGS); the
# PyTorch versions stop after a fixed number of gradient steps. Check: with a larger
# learning rate (fully converged) the manual loop should land on sklearn's solution.
w_conv, b_conv, _ = train_manual(0.5, epochs=5000)
print("converged run (lr=0.5, 5000 epochs) max coef diff vs sklearn:",
      np.abs(w_conv.numpy() - sk_model.coef_[0]).max().round(6))

## 11. Decision threshold — chosen on VALIDATION by business cost
Expected cost = FP x 300k + FN x 600k. Theory says, for well-calibrated probabilities,
the optimum is `COST_FP / (COST_FP + COST_FN)` ≈ 0.33. We pick it empirically on val.

In [ ]:
p_va_sk = sk_model.predict_proba(X_va)[:, 1]
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
val_costs = [evaluate(y_va, p_va_sk, t)["cost_eur"] for t in grid]
BEST_THR = float(grid[int(np.argmin(val_costs))])
print(f"cost-optimal threshold on val: {BEST_THR}  (theory: {COST_FP/(COST_FP+COST_FN):.2f})")

## 12. Final evaluation on TEST (touched once)

In [ ]:
probs_test = {
    "sklearn": sk_model.predict_proba(X_te)[:, 1],
    "torch_manual": manual_predict(w_manual, b_manual, X_te),
    "torch_standard": standard_predict(std_model, X_te),
}
rows = {
    "baseline_majority": evaluate(y_te, p_majority, BEST_THR),
    "baseline_persistence": evaluate(y_te, p_persist, 0.5),
}
for name, p in probs_test.items():
    rows[name] = evaluate(y_te, p, BEST_THR)
results = pd.DataFrame(rows).T
print(results[["roc_auc", "pr_auc", "log_loss", "accuracy", "precision", "recall",
               "cost_eur"]].round(3).to_string())
print("\nmax |p_sklearn - p_manual|  :", np.abs(probs_test["sklearn"] - probs_test["torch_manual"]).max())
print("max |p_sklearn - p_standard|:", np.abs(probs_test["sklearn"] - probs_test["torch_standard"]).max())

# Business baselines: "send to everyone" and "send to no one"
send_all = evaluate(y_te, np.ones(len(y_te)), 0.5)["cost_eur"]
send_none = evaluate(y_te, np.zeros(len(y_te)), 0.5)["cost_eur"]
print(f"\ncost send-to-all:  EUR {send_all:,}")
print(f"cost send-to-none: EUR {send_none:,}")
print(f"model saving vs best naive strategy: "
      f"{1 - rows['sklearn']['cost_eur'] / min(send_all, send_none):.1%}")

# Sensitivity: the model's value depends heavily on the assumed cost ratio
grid_t = np.round(np.arange(0.05, 0.96, 0.01), 2)
sens = []
for ratio in [1, 1.5, 2, 3, 5]:
    c = lambda y, p, t: ((p >= t) & (y == 0)).sum() + ratio * ((p < t) & (y == 1)).sum()
    t_best = grid_t[np.argmin([c(y_va, p_va_sk, t) for t in grid_t])]
    m = c(y_te, probs_test["sklearn"], t_best)
    naive = min(c(y_te, np.ones(len(y_te)), 0.5), c(y_te, np.zeros(len(y_te)), 0.5))
    sens.append({"fn_fp_ratio": ratio, "threshold": t_best, "saving_vs_naive": 1 - m / naive})
sens = pd.DataFrame(sens)
print(sens.round(3).to_string(index=False))

## 13. Ablation — which feature groups actually help? (validation set, sklearn)

In [ ]:
ablation = {}
base_auc = roc_auc_score(y_va, p_va_sk)
for group, cols in FEATURE_GROUPS.items():
    keep = [i for i, f in enumerate(FEATURES)
            if f not in cols and f.removesuffix("_missing") not in cols]
    m = LogisticRegression(C=BEST_C, max_iter=5000).fit(X_tr[:, keep], y_tr)
    auc = roc_auc_score(y_va, m.predict_proba(X_va[:, keep])[:, 1])
    ablation[group] = {"val_auc_without": auc, "auc_drop": base_auc - auc}
ablation = pd.DataFrame(ablation).T
print(f"val AUC with all features: {base_auc:.4f}")
print(ablation.round(4).to_string())

# Robustness check: the Ukraine shock (labels from 2023) vs. the rest of the test set
shock = test["year"] == 2022
for name in ["sklearn"]:
    print(f"\nAUC 2022 rows: {roc_auc_score(y_te[shock], probs_test[name][shock]):.3f}  "
          f"other test years: {roc_auc_score(y_te[~shock], probs_test[name][~shock]):.3f}")
    print(f"mean predicted prob 2022: {probs_test[name][shock].mean():.3f}  "
          f"actual rate: {y_te[shock].mean():.3f}")

## 14. Save artifacts for the dashboard (the app never retrains)

In [ ]:
joblib.dump(sk_model, ART / "sklearn_model.joblib")
# Save the fitted preprocessing *parameters* (plain objects, so app.py can load them
# without needing this notebook's Preprocessor class)
joblib.dump({"low": prep.low, "high": prep.high, "median": prep.median,
             "missing_cols": prep.missing_cols, "columns": prep.columns,
             "numeric": NUMERIC, "region_cols": REGION_COLS, "scaler": prep.scaler},
            ART / "preprocessor.joblib")
torch.save({"w": w_manual, "b": b_manual}, ART / "torch_manual.pt")
torch.save(std_model.state_dict(), ART / "torch_standard.pt")

keep_cols = ["iso3", "country", "region", "year", "target", "next_real_growth"] + NUMERIC


def frame(part, X):
    out = part[keep_cols].copy()
    out["p_sklearn"] = sk_model.predict_proba(X)[:, 1]
    out["p_torch_manual"] = manual_predict(w_manual, b_manual, X)
    out["p_torch_standard"] = standard_predict(std_model, X)
    return out


pd.concat([frame(train, X_tr).assign(split="train"),
           frame(val, X_va).assign(split="val"),
           frame(test, X_te).assign(split="test")]).to_csv(ART / "predictions.csv", index=False)
frame(live, X_live).to_csv(ART / "live_predictions.csv", index=False)
coef.to_csv(ART / "coefficients.csv", index=False)
pd.DataFrame({"epoch": range(EPOCHS), "torch_manual": hist_manual,
              "torch_standard": hist_std}).to_csv(ART / "loss_history.csv", index=False)

json.dump({
    "growth_threshold": GROWTH_THRESHOLD, "cost_fp": COST_FP, "cost_fn": COST_FN,
    "best_C": BEST_C, "best_lr": BEST_LR, "epochs": EPOCHS, "best_threshold": BEST_THR,
    "splits": {"train": TRAIN_YEARS, "val": VAL_YEARS, "test": TEST_YEARS, "live": LIVE_YEAR},
    "feature_groups": FEATURE_GROUPS,
    "test_results": results.to_dict(orient="index"),
    "ablation": ablation.to_dict(orient="index"),
    "cost_send_all": send_all,
    "cost_send_none": send_none,
    "cost_sensitivity": sens.to_dict(orient="records"),
}, open(ART / "metrics.json", "w"), indent=2, default=float)
print("saved:", sorted(p.name for p in ART.iterdir()))